# Notebook 25 — Human-in-the-Loop
## Interactive Agent Systems with Human Oversight

Agents make mistakes. Humans provide **judgment**, **approval**, and **course-correction**.

| Pattern | Description | When to Use |
|---------|-------------|------------|
| Approval Gate | Agent proposes, human approves | High-risk actions |
| Feedback Loop | Human critiques, agent revises | Creative tasks |
| Escalation | Agent detects uncertainty, asks human | Low-confidence situations |
| Interactive Mode | Human can intervene at any step | Debugging, exploration |

**Prerequisites:** Notebooks 21–24 (orchestration, safety).

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Human-in-the-Loop?

Fully autonomous agents fail when:
- Tasks require **judgment** that can't be formalized
- Stakes are high (financial, legal, medical decisions)
- The agent's **confidence is low** and it needs guidance
- **Novel situations** outside training distribution
- **Accountability** requires human sign-off

HITL adds a human to the agent loop — not replacing automation, but **augmenting** it with human judgment at critical points.

```
         ┌──────────┐     ┌──────────┐
Task ──▶ │  Agent   │──▶──│  Human   │──▶ Final Output
         └──────────┘     └──────────┘
              │                │
         (proposes)      (approves/revises)
```

In [ ]:
# Simulated human responses for notebook reproducibility
# In production, these would be actual input() calls or UI interactions
class SimulatedHuman:
    """Simulates human responses for reproducible notebook experiments."""

    def __init__(self, name: str = "Human Reviewer"):
        self.name = name
        self.interactions: List[Dict] = []
        self._response_queue: List[str] = []
        self._default_approval = True

    def set_responses(self, responses: List[str]):
        """Pre-load responses for simulation."""
        self._response_queue = list(responses)

    def get_approval(self, proposal: str) -> bool:
        """Simulate human approval decision."""
        self.interactions.append({"type": "approval", "proposal": proposal[:100]})
        if self._response_queue:
            response = self._response_queue.pop(0)
            approved = response.lower() in ("yes", "y", "approve", "true")
        else:
            approved = self._default_approval
        print(f"  👤 [{self.name}]: {'APPROVED ✓' if approved else 'REJECTED ✗'}")
        return approved

    def get_feedback(self, content: str) -> str:
        """Simulate human feedback."""
        self.interactions.append({"type": "feedback", "content": content[:100]})
        if self._response_queue:
            feedback = self._response_queue.pop(0)
        else:
            feedback = "Looks good, minor improvements needed."
        print(f"  👤 [{self.name}]: {feedback[:80]}")
        return feedback

    def get_choice(self, options: List[str]) -> int:
        """Simulate human choosing from options."""
        self.interactions.append({"type": "choice", "options": options})
        if self._response_queue:
            try:
                choice = int(self._response_queue.pop(0))
            except:
                choice = 0
        else:
            choice = 0
        print(f"  👤 [{self.name}]: Selected option {choice}")
        return min(choice, len(options) - 1)

human = SimulatedHuman("Alice")
print("✓ SimulatedHuman ready (replace with input() for real Colab interaction)")
print("  In Colab, you can replace SimulatedHuman with actual input() calls:"  )
print('  approval = input("Approve? (y/n): ").lower() == "y"')

## 2. Approval Gates — Human Must Approve Before Execution

The agent proposes an action, but **waits for human approval** before executing. Critical for:
- Tool calls with side effects (sending emails, modifying data)
- Actions that are expensive or irreversible
- Decisions with significant consequences

In [ ]:
class ApprovalGate:
    """Gate that requires human approval before proceeding."""

    def __init__(self, human: SimulatedHuman, require_reason: bool = False):
        self.human = human
        self.require_reason = require_reason
        self.approved_count = 0
        self.rejected_count = 0
        self.history: List[Dict] = []

    def request_approval(self, action: str, details: str = "",
                          risk_level: str = "medium") -> Dict[str, Any]:
        """Request human approval for an action."""
        print(f"  🔒 Approval requested [{risk_level.upper()} risk]")
        print(f"     Action: {action}")
        if details:
            print(f"     Details: {details[:100]}")

        approved = self.human.get_approval(f"{action}: {details}")

        if approved:
            self.approved_count += 1
        else:
            self.rejected_count += 1

        entry = {
            "action": action, "details": details[:100],
            "risk_level": risk_level, "approved": approved,
            "timestamp": time.time()
        }
        self.history.append(entry)
        return entry

    def stats(self) -> Dict[str, Any]:
        return {
            "total": self.approved_count + self.rejected_count,
            "approved": self.approved_count,
            "rejected": self.rejected_count,
            "approval_rate": self.approved_count / max(1, self.approved_count + self.rejected_count)
        }


class ApprovalGatedAgent:
    """Agent that requests approval before executing actions."""

    def __init__(self, name: str, gate: ApprovalGate):
        self.name = name
        self.gate = gate
        self.actions_taken: List[Dict] = []

    def propose_and_execute(self, task: str, tools: List[str]) -> Dict[str, Any]:
        """Propose a plan, get approval, then execute."""
        # Step 1: Generate plan
        messages = [
            {"role": "system", "content": f"""You are {self.name}. Given a task, propose ONE specific action.
Format: ACTION: <action_name> | DETAILS: <what you'll do> | RISK: low/medium/high"""},
            {"role": "user", "content": f"Task: {task}\nAvailable tools: {tools}"}
        ]
        proposal = generate(messages, max_new_tokens=100, temperature=0.3, do_sample=True)
        print(f"\n  🤖 [{self.name}] proposes: {proposal[:120]}")

        # Parse proposal
        action = "general_action"
        risk = "medium"
        if "ACTION:" in proposal:
            parts = proposal.split("|")
            for part in parts:
                if "ACTION:" in part:
                    action = part.split("ACTION:")[-1].strip()[:50]
                if "RISK:" in part:
                    risk_text = part.split("RISK:")[-1].strip().lower()
                    if "low" in risk_text: risk = "low"
                    elif "high" in risk_text: risk = "high"

        # Step 2: Get approval
        approval = self.gate.request_approval(action, proposal, risk)

        if approval["approved"]:
            # Step 3: Execute
            exec_messages = [
                {"role": "system", "content": f"You are {self.name}. Execute this action and report the result concisely."},
                {"role": "user", "content": f"Execute: {proposal}"}
            ]
            result = generate(exec_messages, max_new_tokens=200)
            self.actions_taken.append({"action": action, "result": result[:200]})
            return {"status": "executed", "action": action, "result": result}
        else:
            return {"status": "rejected", "action": action, "result": "Action was not approved by human."}

# Demo approval gates
human_reviewer = SimulatedHuman("ReviewerBob")
human_reviewer.set_responses(["yes", "no", "yes"])

gate = ApprovalGate(human_reviewer)
agent = ApprovalGatedAgent("PlannerBot", gate)

print("=" * 70)
print("APPROVAL-GATED AGENT")
print("=" * 70)

tasks = [
    ("Analyze the sales data and create a summary report", ["read_file", "analyze", "write_report"]),
    ("Delete all temporary files older than 30 days", ["list_files", "delete_file"]),
    ("Send a project status email to the team", ["compose_email", "send_email"]),
]

for task, tools in tasks:
    print(f"\nTask: {task}")
    result = agent.propose_and_execute(task, tools)
    print(f"  Status: {result['status']}")

print(f"\nGate stats: {gate.stats()}")

## 3. Feedback Loops — Iterative Refinement with Human Input

The agent generates output, the human provides **text feedback**, and the agent **revises**. This iterative loop continues until the human is satisfied.

In [ ]:
class FeedbackLoop:
    """Iterative generation-feedback-revision loop."""

    def __init__(self, human: SimulatedHuman, max_iterations: int = 3):
        self.human = human
        self.max_iterations = max_iterations
        self.iterations: List[Dict] = []

    def run(self, task: str) -> Dict[str, Any]:
        """Run the feedback loop until approval or max iterations."""
        current_draft = None
        t0 = time.time()

        for i in range(self.max_iterations):
            print(f"\n  --- Iteration {i + 1}/{self.max_iterations} ---")

            # Generate or revise
            if current_draft is None:
                messages = [
                    {"role": "system", "content": "You are an expert writer. Create a first draft. Be thorough but concise (3-5 sentences)."},
                    {"role": "user", "content": task}
                ]
            else:
                feedback = self.human.get_feedback(current_draft)
                messages = [
                    {"role": "system", "content": "You are an expert writer. Revise based on feedback. Keep improvements concise."},
                    {"role": "user", "content": f"Previous draft:\n{current_draft}\n\nFeedback: {feedback}\n\nRevised version:"}
                ]

            draft = generate(messages, max_new_tokens=300)
            current_draft = draft
            print(f"  📝 Draft: {draft[:150]}...")

            self.iterations.append({
                "iteration": i + 1,
                "draft": draft,
                "feedback": None if i == 0 else feedback
            })

            # Check if human approves
            if i < self.max_iterations - 1:
                approved = self.human.get_approval(f"Iteration {i+1} draft")
                if approved:
                    return {
                        "final_draft": current_draft,
                        "iterations": len(self.iterations),
                        "time": round(time.time() - t0, 2),
                        "status": "approved"
                    }

        return {
            "final_draft": current_draft,
            "iterations": len(self.iterations),
            "time": round(time.time() - t0, 2),
            "status": "max_iterations_reached"
        }

# Demo feedback loop
feedback_human = SimulatedHuman("Editor")
feedback_human.set_responses([
    "Make it more specific with concrete examples",  # feedback for draft 1
    "no",  # don't approve yet
    "Add a call to action at the end",  # feedback for draft 2
    "yes",  # approve
])

loop = FeedbackLoop(feedback_human, max_iterations=3)

print("=" * 70)
print("FEEDBACK LOOP — Iterative Refinement")
print("=" * 70)

result = loop.run("Write a compelling introduction for a blog post about using AI agents in software development")

print(f"\n{'=' * 70}")
print(f"Status: {result['status']}")
print(f"Iterations: {result['iterations']}")
print(f"Time: {result['time']}s")
print(f"\nFinal draft:\n{result['final_draft'][:400]}")

## 4. Escalation — Agent Detects Low Confidence, Asks Human

The agent monitors its own **confidence** and escalates to a human when uncertain.

In [ ]:
class EscalationHandler:
    """Handles escalation when agent confidence is low."""

    def __init__(self, human: SimulatedHuman, confidence_threshold: float = 0.6):
        self.human = human
        self.threshold = confidence_threshold
        self.escalations: List[Dict] = []
        self.auto_handled: int = 0

    def assess_confidence(self, task: str, response: str) -> float:
        """Use LLM to self-assess confidence."""
        messages = [
            {"role": "system", "content": "Rate your confidence in this response on a scale of 0.0 to 1.0. Consider accuracy, completeness, and whether you might be wrong. Reply with ONLY a decimal number."},
            {"role": "user", "content": f"Task: {task}\nResponse: {response}"}
        ]
        result = generate(messages, max_new_tokens=10, temperature=0.1, do_sample=False)
        try:
            confidence = float(re.search(r'(0\.\d+|1\.0|0|1)', result).group(1))
        except:
            confidence = 0.5
        return confidence

    def handle(self, task: str) -> Dict[str, Any]:
        """Process task with escalation when confidence is low."""
        # Generate response
        messages = [
            {"role": "system", "content": "You are a helpful assistant. Answer concisely and accurately. If you're unsure, say so."},
            {"role": "user", "content": task}
        ]
        response = generate(messages, max_new_tokens=200)

        # Assess confidence
        confidence = self.assess_confidence(task, response)
        print(f"  Confidence: {confidence:.2f} (threshold: {self.threshold})")

        if confidence >= self.threshold:
            self.auto_handled += 1
            return {
                "response": response,
                "confidence": confidence,
                "escalated": False,
                "source": "agent"
            }
        else:
            # Escalate to human
            print(f"  ⚠ Low confidence — escalating to human")
            self.escalations.append({"task": task, "confidence": confidence})

            human_input = self.human.get_feedback(
                f"Agent unsure about: {task}\nAgent's attempt: {response}"
            )

            # Revise with human guidance
            revise_messages = [
                {"role": "system", "content": "Revise your answer incorporating the human expert's guidance."},
                {"role": "user", "content": f"Task: {task}\nYour draft: {response}\nExpert guidance: {human_input}\n\nRevised answer:"}
            ]
            revised = generate(revise_messages, max_new_tokens=200)

            return {
                "response": revised,
                "original_response": response,
                "confidence": confidence,
                "escalated": True,
                "human_guidance": human_input,
                "source": "agent+human"
            }

# Demo escalation
escalation_human = SimulatedHuman("Expert")
escalation_human.set_responses([
    "The current consensus is that P≠NP but it remains unproven. Focus on practical implications.",
    "Quantum supremacy was demonstrated by Google in 2019, but practical quantum advantage for useful problems is still years away.",
])

handler = EscalationHandler(escalation_human, confidence_threshold=0.6)

tasks = [
    "What is the time complexity of Python's built-in sort?",  # High confidence
    "Is P equal to NP? What is the current consensus?",  # Lower confidence
    "When will quantum computers break RSA encryption?",  # Lower confidence
]

print("=" * 70)
print("ESCALATION HANDLER")
print("=" * 70)

for task in tasks:
    print(f"\nTask: {task}")
    result = handler.handle(task)
    status = "🤖 Auto" if not result["escalated"] else "👤 Escalated"
    print(f"  [{status}] Confidence: {result['confidence']:.2f}")
    print(f"  Response: {result['response'][:150]}...")

print(f"\nStats: {handler.auto_handled} auto-handled, {len(handler.escalations)} escalated")

## 5. Interactive Mode — Human Can Intervene at Any Step

The most hands-on pattern: the human can observe and intervene at **every step** of the agent's execution.

In [ ]:
class InteractiveAgent:
    """Agent where human can observe and intervene at each step."""

    def __init__(self, name: str, human: SimulatedHuman):
        self.name = name
        self.human = human
        self.steps: List[Dict] = []

    def plan(self, task: str) -> List[str]:
        """Generate a multi-step plan."""
        messages = [
            {"role": "system", "content": "Break this task into 3-4 concrete steps. List each step on a new line starting with a number."},
            {"role": "user", "content": task}
        ]
        response = generate(messages, max_new_tokens=200)
        steps = [line.strip() for line in response.split("\n")
                 if line.strip() and line.strip()[0].isdigit()]
        return steps[:4] if steps else [task]

    def execute_step(self, step: str, context: str = "") -> str:
        """Execute a single step."""
        messages = [
            {"role": "system", "content": f"You are {self.name}. Execute this step concisely (2-3 sentences)."},
            {"role": "user", "content": f"Context: {context}\n\nStep to execute: {step}" if context else f"Step: {step}"}
        ]
        return generate(messages, max_new_tokens=200)

    def run_interactive(self, task: str) -> Dict[str, Any]:
        """Run task with human oversight at each step."""
        print(f"\n  📋 Planning: {task[:60]}...")
        plan = self.plan(task)

        print(f"  Generated {len(plan)} steps:")
        for i, step in enumerate(plan):
            print(f"    {i+1}. {step[:70]}")

        # Human can approve/modify the plan
        approved = self.human.get_approval("Execute this plan?")
        if not approved:
            return {"status": "plan_rejected", "plan": plan, "results": []}

        # Execute step by step with human oversight
        results = []
        context = ""
        for i, step in enumerate(plan):
            print(f"\n  Step {i+1}/{len(plan)}: {step[:60]}")

            result = self.execute_step(step, context)
            print(f"  Result: {result[:120]}...")

            # Human can approve, modify, or skip
            step_approved = self.human.get_approval(f"Step {i+1} result")

            if step_approved:
                results.append({"step": step, "result": result, "status": "approved"})
                context += f"\nStep {i+1}: {result[:100]}"
            else:
                # Human provides correction
                correction = self.human.get_feedback(f"Step {i+1}: {result[:100]}")
                results.append({"step": step, "result": correction, "status": "human_corrected"})
                context += f"\nStep {i+1} (human corrected): {correction[:100]}"

            self.steps.append(results[-1])

        return {"status": "completed", "plan": plan, "results": results}

# Demo interactive agent
interactive_human = SimulatedHuman("Supervisor")
interactive_human.set_responses([
    "yes",     # approve plan
    "yes",     # approve step 1
    "no",      # reject step 2
    "Focus more on practical implementation details rather than theory",  # correction
    "yes",     # approve step 3
    "yes",     # approve step 4
])

agent = InteractiveAgent("StepBot", interactive_human)

print("=" * 70)
print("INTERACTIVE AGENT — Step-by-Step with Human Oversight")
print("=" * 70)

result = agent.run_interactive("Create a plan for implementing a REST API with authentication")

print(f"\n{'=' * 70}")
print(f"Status: {result['status']}")
print(f"Steps executed: {len(result['results'])}")
for r in result['results']:
    print(f"  [{r['status']}] {r['step'][:50]}")

## 6. Experiment — Approval-Gated Tool Use

Agent must request permission before **each tool call**. This prevents unintended side effects.

In [ ]:
class ApprovalGatedToolUser:
    """Agent that requests approval before each tool call."""

    def __init__(self, human: SimulatedHuman):
        self.human = human
        self.tools = {
            "web_search": lambda q: f"Search results for '{q}': [Result 1: overview...] [Result 2: tutorial...]",
            "read_file": lambda f: f"Contents of {f}: [sample data...]",
            "write_file": lambda f: f"Wrote to {f}",
            "run_code": lambda c: f"Output of code: {c[:30]}... → [result]",
            "send_notification": lambda m: f"Notification sent: {m[:50]}",
        }
        self.approved_calls: List[Dict] = []
        self.rejected_calls: List[Dict] = []

    def call_tool(self, tool_name: str, argument: str) -> Dict[str, Any]:
        """Request approval then call tool."""
        print(f"\n  🔧 Tool request: {tool_name}({argument[:40]})")

        if tool_name not in self.tools:
            return {"status": "error", "result": f"Unknown tool: {tool_name}"}

        approved = self.human.get_approval(f"Call {tool_name}({argument[:40]})?")

        if approved:
            result = self.tools[tool_name](argument)
            self.approved_calls.append({"tool": tool_name, "arg": argument[:50]})
            return {"status": "executed", "result": result}
        else:
            self.rejected_calls.append({"tool": tool_name, "arg": argument[:50]})
            return {"status": "rejected", "result": f"Tool call rejected by human"}

    def run_task(self, task: str) -> Dict[str, Any]:
        """Plan and execute a task with approval-gated tools."""
        messages = [
            {"role": "system", "content": f"""You are a helpful assistant with tools: {list(self.tools.keys())}.
For the given task, list 3-4 tool calls you'd make. Format each as: TOOL: name | ARG: argument"""},
            {"role": "user", "content": task}
        ]
        plan = generate(messages, max_new_tokens=200)
        print(f"  📋 Plan: {plan[:200]}")

        # Parse and execute tool calls
        results = []
        for line in plan.split("\n"):
            if "TOOL:" in line and "ARG:" in line:
                parts = line.split("|")
                tool = parts[0].split("TOOL:")[-1].strip().lower().replace(" ", "_")
                arg = parts[1].split("ARG:")[-1].strip() if len(parts) > 1 else ""
                # Match to closest tool name
                matched_tool = None
                for t in self.tools:
                    if t in tool or tool in t:
                        matched_tool = t
                        break
                if matched_tool:
                    result = self.call_tool(matched_tool, arg)
                    results.append(result)

        return {"plan": plan, "tool_results": results,
                "approved": len(self.approved_calls), "rejected": len(self.rejected_calls)}

# Demo
tool_human = SimulatedHuman("ToolReviewer")
tool_human.set_responses(["yes", "yes", "no", "yes"])

tool_agent = ApprovalGatedToolUser(tool_human)

print("=" * 70)
print("APPROVAL-GATED TOOL USE")
print("=" * 70)

result = tool_agent.run_task("Research Python best practices, save a summary, and notify the team")

print(f"\n{'=' * 70}")
print(f"Tool calls approved: {result['approved']}")
print(f"Tool calls rejected: {result['rejected']}")

## 7. Experiment — Feedback-Driven Writing

Agent writes a piece, human critiques it, agent revises. Multiple rounds of refinement.

In [ ]:
class FeedbackDrivenWriter:
    """Agent that writes and revises based on human feedback."""

    def __init__(self, human: SimulatedHuman, max_rounds: int = 3):
        self.human = human
        self.max_rounds = max_rounds
        self.drafts: List[Dict] = []

    def write(self, task: str) -> Dict[str, Any]:
        """Write with iterative human feedback."""
        t0 = time.time()
        current_draft = None

        for round_num in range(1, self.max_rounds + 1):
            print(f"\n  --- Round {round_num} ---")

            if current_draft is None:
                # First draft
                messages = [
                    {"role": "system", "content": "You are an expert writer. Write a clear, engaging piece. 4-6 sentences."},
                    {"role": "user", "content": task}
                ]
            else:
                # Revision based on feedback
                feedback = self.human.get_feedback(current_draft)
                messages = [
                    {"role": "system", "content": "Revise the text based on feedback. Maintain quality while addressing all feedback points."},
                    {"role": "user", "content": f"Current draft:\n{current_draft}\n\nFeedback: {feedback}\n\nRevised version:"}
                ]

            draft = generate(messages, max_new_tokens=300)
            current_draft = draft
            self.drafts.append({"round": round_num, "draft": draft})
            print(f"  📝 {draft[:150]}...")

            # Check satisfaction
            if round_num < self.max_rounds:
                satisfied = self.human.get_approval(f"Satisfied with round {round_num}?")
                if satisfied:
                    break

        return {
            "final": current_draft,
            "rounds": len(self.drafts),
            "time": round(time.time() - t0, 2),
            "all_drafts": self.drafts
        }

# Demo
writer_human = SimulatedHuman("Editor")
writer_human.set_responses([
    "Add more specific examples and make the tone more conversational",  # feedback 1
    "no",    # not satisfied yet
    "The examples are good now, but tighten up the conclusion",  # feedback 2
    "yes",   # satisfied
])

writer = FeedbackDrivenWriter(writer_human, max_rounds=3)

print("=" * 70)
print("FEEDBACK-DRIVEN WRITING")
print("=" * 70)

result = writer.write("Write an engaging paragraph explaining why code review is important for software teams")

print(f"\n{'=' * 70}")
print(f"Rounds: {result['rounds']} | Time: {result['time']}s")
print(f"\nFinal version:\n{result['final'][:500]}")

# Show draft evolution
print(f"\nDraft evolution:")
for d in result['all_drafts']:
    print(f"  Round {d['round']}: {d['draft'][:80]}...")

## 8. Comparison — Autonomous vs Approval-Gated vs Interactive

Same task, three levels of human involvement. Which produces the best results?

In [ ]:
# Run the same task with different HITL levels
comparison_task = "Suggest three strategies for improving code quality in a software team"

print("=" * 70)
print("COMPARISON: Three Levels of Human Involvement")
print("=" * 70)

# Level 1: Fully Autonomous
print("\n--- Level 1: FULLY AUTONOMOUS ---")
t0 = time.time()
auto_messages = [
    {"role": "system", "content": "You are an expert software engineering consultant. Be concise and practical."},
    {"role": "user", "content": comparison_task}
]
auto_result = generate(auto_messages, max_new_tokens=300)
auto_time = time.time() - t0
print(f"  Time: {auto_time:.1f}s")
print(f"  Result: {auto_result[:200]}...")

# Level 2: Approval-Gated
print("\n--- Level 2: APPROVAL-GATED ---")
approval_human = SimulatedHuman("Approver")
approval_human.set_responses(["yes"])
approval_gate = ApprovalGate(approval_human)

t0 = time.time()
gate_messages = [
    {"role": "system", "content": "You are an expert software engineering consultant. Be concise and practical."},
    {"role": "user", "content": comparison_task}
]
gate_result = generate(gate_messages, max_new_tokens=300)
approval = approval_gate.request_approval("Deliver response", gate_result[:100])
gate_time = time.time() - t0
print(f"  Time: {gate_time:.1f}s (includes approval)")
print(f"  Result: {gate_result[:200]}...")

# Level 3: Interactive with Feedback
print("\n--- Level 3: INTERACTIVE WITH FEEDBACK ---")
interactive_human = SimulatedHuman("Coach")
interactive_human.set_responses([
    "Add concrete tools/practices for each strategy",  # feedback
    "yes",  # approve
])

feedback_loop = FeedbackLoop(interactive_human, max_iterations=2)
t0 = time.time()
interactive_result = feedback_loop.run(comparison_task)
interactive_time = time.time() - t0
print(f"  Time: {interactive_time:.1f}s")
print(f"  Iterations: {interactive_result['iterations']}")

In [ ]:
# Summary comparison
print("=" * 70)
print("HITL COMPARISON SUMMARY")
print("=" * 70)
print(f"""
Level              │ Time    │ Human Effort │ Quality Control │ Best For
───────────────────┼─────────┼──────────────┼─────────────────┼──────────────────
1. Autonomous      │ {auto_time:>5.1f}s  │ None         │ None            │ Low-risk, routine
2. Approval-gated  │ {gate_time:>5.1f}s  │ 1 decision   │ Pass/fail       │ Medium-risk actions
3. Interactive     │ {interactive_time:>5.1f}s  │ Multi-turn   │ Iterative       │ High-stakes, creative

Trust spectrum:
  Full automation ◄──────────────────────────────► Full human control
  Fast, scalable                           Slow, high quality
  Risk of errors                           Human bottleneck

Recommendation:
  • Start with approval gates for new agent deployments
  • Move to autonomous mode as trust builds
  • Keep interactive mode for high-stakes or novel tasks
  • Log everything for continuous improvement
""")

## 9. Key Takeaways

### HITL Patterns
1. **Approval Gates** — simple pass/fail before execution; good for tool calls with side effects
2. **Feedback Loops** — iterative refinement; best for creative/writing tasks
3. **Escalation** — agent self-assesses confidence; efficient use of human time
4. **Interactive Mode** — maximum control; human supervises every step

### Design Principles
- **Progressive autonomy** — start supervised, gradually increase automation as trust builds
- **Selective intervention** — humans should focus on high-value decisions, not routine checks
- **Graceful degradation** — system should work (at reduced speed) even without human availability
- **Audit trail** — log every human interaction for learning and accountability

### When to Use Each Pattern
| Pattern | Human Effort | Best For |
|---------|-------------|----------|
| Approval Gate | Low (yes/no) | Tool calls, deployments |
| Feedback Loop | Medium (text input) | Writing, creative tasks |
| Escalation | Variable (on-demand) | Knowledge-intensive work |
| Interactive | High (per-step) | Debugging, novel tasks |

### Production Considerations
- Use **timeouts** — don't block indefinitely waiting for human input
- Implement **delegation** — if the primary human is unavailable, escalate to a backup
- Track **approval latency** — identify bottlenecks in the human review process
- Use **batch review** — group similar low-risk items for efficient human review

### Course Progress
This completes the **Advanced Agent Patterns** section:
- ✅ Notebook 21: Orchestration patterns
- ✅ Notebook 22: Shared state & blackboard
- ✅ Notebook 23: Swarm intelligence
- ✅ Notebook 24: Safety & guardrails
- ✅ Notebook 25: Human-in-the-loop